In [ ]:
# =============================================================
# 06_task_B_trajectory_classification.ipynb
# Task B — Trajectory Classification
# ------------------------------------------------------------
# Benchmark task: classify each country into one of four
# trajectory classes (EE, AE, IR, DL) using panel-level
# aggregated features — i.e. summary statistics computed
# over the full 2015–2024 observation window.
#
# Problem formulation
# ───────────────────
# Input:  country-level feature vectors derived from the panel
#         (means, trends, missingness rates per indicator)
# Output: one of {EE, AE, IR, DL} per country
# Labels: from trajectory_labels.csv (Step 3)
# N:      12 countries (small-n classification)
#
# Evaluation protocol
# ───────────────────
# Leave-One-Country-Out (LOCO) cross-validation only.
# With N=12, a single temporal train/test split would leave
# only 3 test countries — insufficient for any metric.
# LOCO gives 12 folds, each with 1 test country.
# Macro-F1, per-class F1, and accuracy are reported.
#
# Models
# ──────
# 1. Majority-class baseline
# 2. Logistic Regression (L2, multinomial)
# 3. Linear SVC (one-vs-rest)
# 4. Random Forest classifier
# 5. k-NN (k=3, Euclidean distance)
#
# Feature engineering
# ───────────────────
# Per country, compute:
#   - mean and std of each indicator over 2015–2024
#   - linear trend slope (OLS) per indicator
#   - missingness rate per indicator
#   - trajectory delta features (ΔGini, ΔTertiary, ΔCompletion)
#
# Outputs
# ───────
#   outputs/results/task_B_metrics.csv
#   outputs/results/task_B_predictions.csv
#   outputs/tables/table_taskB.csv / .tex
#   outputs/figures/panel_12/taskB_confusion_matrix.png
#   outputs/figures/panel_12/taskB_macro_f1_comparison.png
#   outputs/figures/panel_12/taskB_feature_heatmap.png
#
# Usage
# ─────
#   python src/06_task_B_trajectory_classification.py
#
# Requirements
# ────────────
#   pip install scikit-learn pandas numpy matplotlib pyyaml
# =============================================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score,
    confusion_matrix, classification_report,
)
from sklearn.impute import SimpleImputer

from utils import find_project_root, load_indicators, load_pipeline

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

ALL_INDICATORS = list(IND_CFG["indicators"].keys())

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
RESULT_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR  = PROJECT_ROOT / "outputs" / "tables"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

for d in [RESULT_DIR, TABLE_DIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_ORDER  = ["EE", "AE", "IR", "DL"]
CLASS_NAMES  = {
    "EE": "Equity Expander",
    "AE": "Access Expander",
    "IR": "Inequality Reducer",
    "DL": "Data-Limited",
}
CLASS_COLOURS = {
    "EE": "#1a7a3a",
    "AE": "#1a6fad",
    "IR": "#d73027",
    "DL": "#888888",
}

print(f"[info] Project root : {PROJECT_ROOT}")
print(f"[info] Year range   : {YEAR_MIN}–{YEAR_MAX}")
print(f"[info] Classes      : {CLASS_ORDER}")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    panel_path = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    label_path = DATA_DIR / "trajectory_labels.csv"

    for p in [panel_path, label_path]:
        if not p.exists():
            raise FileNotFoundError(f"Required: {p}")

    panel  = pd.read_csv(panel_path)
    labels = pd.read_csv(label_path)

    print(f"\n[step 0] Loaded panel : {len(panel)} rows")
    print(f"         Labels       : {len(labels)} countries")
    print(f"         Class dist.  : "
          + ", ".join(f"{c}={n}" for c, n in
                      labels["label_code"].value_counts()
                      .reindex(CLASS_ORDER).fillna(0)
                      .astype(int).items()))
    return panel, labels


# ──────────────────────────────────────────────────────────────
# Feature engineering — country-level summary vectors
# ──────────────────────────────────────────────────────────────
def _trend_slope(series: pd.Series) -> float:
    """OLS slope of series against its integer index (0,1,2,…)."""
    valid = series.dropna()
    if len(valid) < 2:
        return np.nan
    x = np.arange(len(valid), dtype=float)
    x -= x.mean()
    y = valid.values - valid.mean()
    denom = (x * x).sum()
    return float((x * y).sum() / denom) if denom > 0 else 0.0


def build_country_features(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate panel to one feature vector per country.

    For each indicator:
      {ind}_mean   — temporal mean
      {ind}_std    — temporal std
      {ind}_trend  — OLS slope per year
      {ind}_miss   — missingness rate (fraction NaN)

    Plus the raw delta features used for labelling:
      delta_gini, delta_tert, delta_comp
      gini_coverage
    """
    ind_cols = [c for c in ALL_INDICATORS if c in panel.columns]
    records  = []

    for iso, grp in panel.groupby("iso3c"):
        grp   = grp.sort_values("year")
        row   = {"iso3c": iso,
                 "country": grp["country"].iloc[0],
                 "income_group": grp["income_group"].iloc[0]}

        for ind in ind_cols:
            s = grp[ind]
            row[f"{ind}_mean"]  = float(s.mean())   if s.notna().any() else np.nan
            row[f"{ind}_std"]   = float(s.std())    if s.notna().sum() >= 2 else np.nan
            row[f"{ind}_trend"] = _trend_slope(s)
            row[f"{ind}_miss"]  = float(s.isna().mean())

        records.append(row)

    feat_df = pd.DataFrame(records)
    print(f"\n[step 1] Country feature matrix: "
          f"{len(feat_df)} countries × "
          f"{len(feat_df.columns) - 3} features")
    return feat_df


def get_feature_cols(feat_df: pd.DataFrame) -> list[str]:
    """Return all numeric feature column names (exclude identifiers)."""
    exclude = {"iso3c", "country", "income_group"}
    return [c for c in feat_df.columns if c not in exclude
            and feat_df[c].dtype in [np.float64, np.float32, float]]


# ──────────────────────────────────────────────────────────────
# Model registry
# ──────────────────────────────────────────────────────────────
def get_models() -> dict:
    return {
        "Majority_Baseline": None,
        "LogisticRegression": LogisticRegression(
            C=0.1, max_iter=1000, random_state=42,
            solver="lbfgs",
        ),
        "LinearSVC": LinearSVC(
            C=0.1, max_iter=2000, random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=500, max_depth=3,
            min_samples_leaf=1, random_state=42, n_jobs=-1,
        ),
        "kNN_k3": KNeighborsClassifier(n_neighbors=3),
    }


# ──────────────────────────────────────────────────────────────
# LOCO cross-validation
# ──────────────────────────────────────────────────────────────
def run_loco(
    feat_df:      pd.DataFrame,
    labels:       pd.DataFrame,
    feature_cols: list[str],
    models:       dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Leave-One-Country-Out CV.

    Returns
    -------
    pred_df   : fold-level predictions (iso3c, true_label, model, pred_label)
    metric_df : per-fold + aggregate metrics
    """
    # Merge labels into feature df
    df = feat_df.merge(
        labels[["iso3c", "label_code"]], on="iso3c", how="left"
    )

    countries    = df["iso3c"].tolist()
    all_preds    = []
    fold_metrics = []

    print(f"\n[step 2] LOCO cross-validation ({len(countries)} folds) …")

    for fold, held_iso in enumerate(countries):
        train_df = df[df["iso3c"] != held_iso]
        test_df  = df[df["iso3c"] == held_iso]

        X_tr_raw = train_df[feature_cols].values.astype(float)
        X_te_raw = test_df[feature_cols].values.astype(float)
        y_tr     = train_df["label_code"].values
        y_te     = test_df["label_code"].values

        # Impute + scale on train
        imp    = SimpleImputer(strategy="median")
        X_tr   = imp.fit_transform(X_tr_raw)
        X_te   = imp.transform(X_te_raw)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        majority_class = (
            pd.Series(y_tr).value_counts().idxmax()
        )

        for name, model in models.items():
            if model is None:
                y_pred = np.array([majority_class] * len(y_te))
            else:
                from sklearn.base import clone as sk_clone
                m = sk_clone(model)
                # Use scaled features for linear models, raw for tree/kNN
                use_scaled = name in ("LogisticRegression", "LinearSVC")
                m.fit(X_tr_s if use_scaled else X_tr,
                      y_tr)
                y_pred = m.predict(X_te_s if use_scaled else X_te)

            correct = int(y_pred[0] == y_te[0])
            all_preds.append({
                "fold":       fold,
                "iso3c":      held_iso,
                "country":    test_df["country"].iloc[0],
                "true_label": y_te[0],
                "pred_label": y_pred[0],
                "correct":    correct,
                "model":      name,
            })

    pred_df = pd.DataFrame(all_preds)

    # Aggregate metrics per model
    print(f"\n  Aggregate LOCO metrics (N=12):")
    print(f"  {'Model':<22}  {'Accuracy':>9}  {'Macro-F1':>9}  "
          f"{'EE-F1':>7}  {'AE-F1':>7}  {'IR-F1':>7}  {'DL-F1':>7}")
    print(f"  {'-'*22}  {'-'*9}  {'-'*9}  "
          f"{'-'*7}  {'-'*7}  {'-'*7}  {'-'*7}")

    metric_rows = []
    for name in models.keys():
        sub = pred_df[pred_df["model"] == name]
        y_true = sub["true_label"].values
        y_pred = sub["pred_label"].values

        acc   = accuracy_score(y_true, y_pred)
        # Use zero_division=0 — some classes may never be predicted
        mf1   = f1_score(y_true, y_pred, average="macro",
                         labels=CLASS_ORDER, zero_division=0)
        f1s   = f1_score(y_true, y_pred, average=None,
                         labels=CLASS_ORDER, zero_division=0)
        f1_per = dict(zip(CLASS_ORDER, f1s))

        print(f"  {name:<22}  {acc:>9.3f}  {mf1:>9.3f}  "
              + "  ".join(f"{f1_per.get(c,0):.3f}".rjust(7)
                          for c in CLASS_ORDER))

        metric_rows.append({
            "model":    name,
            "accuracy": round(acc, 4),
            "macro_f1": round(mf1, 4),
            **{f"f1_{c}": round(f1_per.get(c, 0), 4) for c in CLASS_ORDER},
        })

    metric_df = pd.DataFrame(metric_rows)
    return pred_df, metric_df


# ──────────────────────────────────────────────────────────────
# Figures
# ──────────────────────────────────────────────────────────────
def plot_confusion_matrix(pred_df: pd.DataFrame) -> None:
    """
    Grid of confusion matrices, one per model.
    Rows = true class, cols = predicted class.
    """
    model_names = [m for m in pred_df["model"].unique()
                   if m != "Majority_Baseline"]
    n_models    = len(model_names)
    ncols       = min(n_models, 4)
    nrows       = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.5 * ncols, 4 * nrows),
        gridspec_kw={"hspace": 0.5, "wspace": 0.35},
    )
    axes = np.array(axes).flatten()

    for ax_i, name in enumerate(model_names):
        ax  = axes[ax_i]
        sub = pred_df[pred_df["model"] == name]
        cm  = confusion_matrix(
            sub["true_label"], sub["pred_label"],
            labels=CLASS_ORDER,
        )
        mf1 = f1_score(sub["true_label"], sub["pred_label"],
                       average="macro", labels=CLASS_ORDER,
                       zero_division=0)
        acc = accuracy_score(sub["true_label"], sub["pred_label"])

        im = ax.imshow(cm, interpolation="nearest",
                       cmap="Blues", vmin=0, vmax=cm.max() or 1)

        ax.set_xticks(range(len(CLASS_ORDER)))
        ax.set_yticks(range(len(CLASS_ORDER)))
        ax.set_xticklabels(CLASS_ORDER, fontsize=9)
        ax.set_yticklabels(CLASS_ORDER, fontsize=9)
        ax.set_xlabel("Predicted", fontsize=9)
        ax.set_ylabel("True",      fontsize=9)
        ax.set_title(
            f"{name}\nAcc={acc:.2f}  Macro-F1={mf1:.2f}",
            fontsize=9, fontweight="bold",
        )

        # Annotate cells
        thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
        for i in range(len(CLASS_ORDER)):
            for j in range(len(CLASS_ORDER)):
                ax.text(j, i, str(cm[i, j]),
                        ha="center", va="center", fontsize=12,
                        color="white" if cm[i, j] > thresh else "black")

    # Hide unused axes
    for ax_i in range(len(model_names), len(axes)):
        axes[ax_i].set_visible(False)

    fig.suptitle(
        f"Task B — Confusion matrices (LOCO, N=12)\n"
        f"Rows=True class, Cols=Predicted class",
        fontsize=12, fontweight="bold", y=1.01,
    )
    out = FIGDIR / f"taskB_confusion_matrix_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_macro_f1(metric_df: pd.DataFrame) -> None:
    """
    Grouped bar chart: per-class F1 and macro-F1 per model.
    """
    models = metric_df["model"].tolist()
    x      = np.arange(len(models))
    width  = 0.15

    fig, ax = plt.subplots(figsize=(13, 5))

    bar_cols = [f"f1_{c}" for c in CLASS_ORDER] + ["macro_f1"]
    bar_labels = [CLASS_NAMES[c] for c in CLASS_ORDER] + ["Macro-F1"]
    bar_colours = [CLASS_COLOURS[c] for c in CLASS_ORDER] + ["#333333"]

    for i, (col, label, colour) in enumerate(
        zip(bar_cols, bar_labels, bar_colours)
    ):
        offset = (i - len(bar_cols) / 2 + 0.5) * width
        vals   = metric_df[col].values
        bars   = ax.bar(x + offset, vals, width,
                        label=label, color=colour,
                        alpha=0.85, edgecolor="white")
        for bar in bars:
            h = bar.get_height()
            if h > 0.02:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                        f"{h:.2f}", ha="center", va="bottom",
                        fontsize=6.5, rotation=45)

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("F1 score", fontsize=10)
    ax.set_title(
        f"Task B — Classification F1 scores (LOCO, N=12)",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=8, loc="upper right", frameon=True,
              edgecolor="#cccccc", ncol=2)
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.axhline(1/len(CLASS_ORDER), color="#aaaaaa", lw=0.8, ls="--",
               label="Random chance")
    ax.text(len(models) - 0.5, 1/len(CLASS_ORDER) + 0.01,
            "Random\nchance", ha="right", fontsize=7, color="#aaaaaa")

    fig.tight_layout()
    out = FIGDIR / f"taskB_macro_f1_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_feature_heatmap(
    feat_df:      pd.DataFrame,
    labels:       pd.DataFrame,
    feature_cols: list[str],
) -> None:
    """
    Heatmap of standardised mean features per country,
    annotated with trajectory class.
    Countries sorted by class.
    """
    df = feat_df.merge(labels[["iso3c","label_code"]], on="iso3c")
    df = df.sort_values("label_code")

    # Select top-20 most variable features for legibility
    variances = feat_df[feature_cols].var()
    top_feats = variances.nlargest(20).index.tolist()

    X = df[top_feats].values.astype(float)
    imp = SimpleImputer(strategy="median")
    X   = imp.fit_transform(X)
    sc  = StandardScaler()
    X_s = sc.fit_transform(X)

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(X_s, aspect="auto", interpolation="nearest",
                   cmap="RdBu_r", vmin=-3, vmax=3)

    # Country labels with class colour
    ytick_labels = [
        f"{row['iso3c']} ({row['label_code']})"
        for _, row in df.iterrows()
    ]
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(ytick_labels, fontsize=9)
    for i, (_, row) in enumerate(df.iterrows()):
        ax.get_yticklabels()[i].set_color(
            CLASS_COLOURS.get(row["label_code"], "black")
        )

    # Feature labels (strip suffix for brevity)
    short_feats = [
        f.replace("_mean","_μ").replace("_trend","_β")
         .replace("_std","_σ").replace("_miss","_∅")
         .replace("sec_completion","comp")
         .replace("tert_enrol_gross","tert")
         .replace("gini","gini")
         .replace("gpi_sec","gpi_s")
         .replace("gpi_tert","gpi_t")
         .replace("edu_spend_gdp","edu$")
         .replace("gdp_pc_ppp","gdp")
        for f in top_feats
    ]
    ax.set_xticks(range(len(top_feats)))
    ax.set_xticklabels(short_feats, rotation=45, ha="right", fontsize=8)

    cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label("Standardised value", fontsize=9)

    # Class legend
    handles = [mpatches.Patch(color=CLASS_COLOURS[c],
                               label=f"{c} — {CLASS_NAMES[c]}")
               for c in CLASS_ORDER]
    ax.legend(handles=handles, loc="upper left",
              bbox_to_anchor=(1.08, 1.0), fontsize=8,
              frameon=True, edgecolor="#cccccc")

    ax.set_title(
        "Task B — Standardised country feature profiles "
        "(top-20 by variance)\nColour = trajectory class",
        fontsize=10, fontweight="bold",
    )
    fig.tight_layout()
    out = FIGDIR / f"taskB_feature_heatmap_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────────────
def save_results(
    pred_df:   pd.DataFrame,
    metric_df: pd.DataFrame,
) -> None:
    print("\n[step 4] Saving results …")

    # Full predictions
    pred_df.to_csv(RESULT_DIR / "task_B_predictions.csv", index=False)
    print(f"  [OK] task_B_predictions.csv  ({len(pred_df)} rows)")

    # Metrics
    metric_df.to_csv(RESULT_DIR / "task_B_metrics.csv", index=False)
    print(f"  [OK] task_B_metrics.csv")

    # LaTeX table
    metric_df.to_csv(TABLE_DIR / "table_taskB.csv", index=False)
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Task B — Trajectory classification results "
        r"(LOCO, $N=12$). Per-class F1 and macro-F1 reported. "
        r"EE=Equity Expander ($n$=7), AE=Access Expander ($n$=2), "
        r"IR=Inequality Reducer ($n$=2), DL=Data-Limited ($n$=1).}",
        r"\label{tab:taskB}",
        r"\begin{tabular}{lrrrrrr}",
        r"\toprule",
        r"Model & Acc. & Macro-F1 & F1$_{\text{EE}}$ & "
        r"F1$_{\text{AE}}$ & F1$_{\text{IR}}$ & "
        r"F1$_{\text{DL}}$ \\",
        r"\midrule",
    ]
    for _, row in metric_df.iterrows():
        def f(v): return f"{v:.3f}" if pd.notna(v) else "--"
        lines.append(
            f"{row['model']} & {f(row['accuracy'])} & "
            f"{f(row['macro_f1'])} & {f(row['f1_EE'])} & "
            f"{f(row['f1_AE'])} & {f(row['f1_IR'])} & "
            f"{f(row['f1_DL'])} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABLE_DIR / "table_taskB.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    print(f"  [OK] table_taskB.csv / .tex")
    print(f"\n[done] Results : {RESULT_DIR.resolve()}")
    print(f"[done] Tables  : {TABLE_DIR.resolve()}")
    print(f"[done] Figures : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/07_task_C_fairness_optimisation.py")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, labels = load_data()

    print("\n[step 1] Building country feature vectors …")
    feat_df      = build_country_features(panel)
    feature_cols = get_feature_cols(feat_df)

    models = get_models()
    print(f"\n[step 2] Models: {list(models.keys())}")

    pred_df, metric_df = run_loco(feat_df, labels, feature_cols, models)

    print("\n[step 3] Figures …")
    plot_confusion_matrix(pred_df)
    plot_macro_f1(metric_df)
    plot_feature_heatmap(feat_df, labels, feature_cols)

    save_results(pred_df, metric_df)


if __name__ == "__main__":
    main()